# Generative Gaussian Models 

Implementation of all the requests in the project section for the binary project dataset:

- MVG
- Tied Gaussian
- Naive Bayes Gaussian
- Covariance and Correlation analysis
- Feature-subset experiments
- PCA as pre-processing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy import linalg as la
import scipy.linalg

## Helper functions

The functions below keep separate the main parts of the pipeline: 
- parameter estimation
- LLR computation (Log-Likelihood Ratio)
- Prediction from LLR
- Error-rate computation

In [ ]:
def vcol(x):
    """
        Convert the data to Column Vector
    """
    return x.reshape((x.size, 1))

def vrow(x):
    """
        Convert the data to Row Vector
    """
    return x.reshape((1, x.size))


def load_project_data(path):

    """
        Loads the Project Data
    """
    D = []
    L = []
    with open(path, 'r') as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            D.append([float(x) for x in parts[:-1]])
            L.append(int(parts[-1]))
    return np.array(D).T, np.array(L, dtype=np.int32)



def split_db(D, L, seed=0, ratio=2.0 / 3.0):

    """
        Splits the data to training/validation datasets
    """

    nTrain = int(D.shape[1] * ratio)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    idxTrain = idx[:nTrain]
    idxVal = idx[nTrain:]
    return (D[:, idxTrain], L[idxTrain]), (D[:, idxVal], L[idxVal])


def compute_mu_C(D):

    """
        Compute Mean & Covariance Matrix for each class
         - D: The data of one class
         - mu: Mean Average
         - C: Covariance Matrix
    """
    mu = vcol(D.mean(1))
    DC = D - mu
    C = (DC @ DC.T) / D.shape[1]
    return mu, C


def logpdf_GAU_ND(X, mu, C):

    """
        log-probability density function (log-pdf)
        
        Computing the log of probability density function (pdf)
    """
    XC = X - mu
    M = X.shape[0]
    sign, logdet = np.linalg.slogdet(C)
    invC = np.linalg.inv(C)
    return -0.5 * M * np.log(2 * np.pi) - 0.5 * logdet - 0.5 * ((XC) * (invC @ XC)).sum(0)




def mvg_params(DTR, LTR):
    """
        Dictionary Comprehension
            - Isolating the data for each class and calculate
                - Mean
                - Covariance
    """

    """
        for c in sorte(set(LTR)):
            class_parameter_tupel = compute(DTR[:, LTR == c])
            return {
                c: class_parameter_tupel 
            }
    """
    return {c: compute_mu_C(DTR[:, LTR == c]) for c in sorted(set(LTR))}



def tied_params(DTR, LTR):

    """
    - calculates the unique mean for every class
    - forces all classes to share global, averaged covariance matrix 
    
    """
    means = {}
    C_tied = 0
    for c in sorted(set(LTR)):
        Dc = DTR[:, LTR == c]
        mu, Cc = compute_mu_C(Dc)
        means[c] = mu
        C_tied += Cc * Dc.shape[1]
    C_tied /= DTR.shape[1]
    return {c: (means[c], C_tied) for c in sorted(means)}



def naive_params(DTR, LTR):
    """
        Creates an Identity Matrix 
            - matrix of 0s, with 1s down the main diagonal

        Multiplying the full covariance matrix by this identity matrix
    """
    out = {}
    for c in sorted(set(LTR)):
        mu, C = compute_mu_C(DTR[:, LTR == c])
        out[c] = (mu, C * np.eye(DTR.shape[0]))
    return out


def compute_llr(D, params):
    """
        Computes the Log-Likelihood Ratio (LLR) 
        for a binary classification task.

        Likelihood ratio is the probability of 
            Class 1 divided by the probability of Class 0.
    
    """
    mu1, C1 = params[1]
    mu0, C0 = params[0]
    return logpdf_GAU_ND(D, mu1, C1) - logpdf_GAU_ND(D, mu0, C0)



def predict_from_llr(llr, threshold=0):
    
    """
        Compares the computed LLR scores
        against a specific threshold to 
        assign the final class labels (0 or 1)

        if LLR is positive, it means Class 1 is more likely
        so llr >= threshold will return True and python
        is going to convert to 1
    """
    return np.int32(llr >= threshold)

def compute_error_rate(pred, labels):

    """
        Calculates percentage of total mistakes the model made.
    """
    return (pred != labels).mean()

def evaluate_model(DTR, LTR, DVAL, LVAL, model_type, threshold=0):
    
    if model_type == 'MVG':
        params = mvg_params(DTR, LTR)
    elif model_type == 'Tied':
        params = tied_params(DTR, LTR)
    elif model_type == 'Naive':
        params = naive_params(DTR, LTR)
    else:
        raise ValueError('Unknown model type')
    
    llr = compute_llr(DVAL, params)
    pred = predict_from_llr(llr, threshold=threshold)
    err = compute_error_rate(pred, LVAL)
    return params, llr, pred, err

def compute_pca(D, m):
    mu = vcol(D.mean(1))
    DC = D - mu
    C = (DC @ DC.T) / D.shape[1]
    U, s, Vh = np.linalg.svd(C)
    return U[:, :m]

def apply_pca(P, D):
    return P.T @ D

def compute_lda(D, L, m=1):
    classes = sorted(set(L))
    mu = vcol(D.mean(1))
    SW = 0
    SB = 0
    for c in classes:
        Dc = D[:, L == c]
        muc = vcol(Dc.mean(1))
        DcC = Dc - muc
        SW += DcC @ DcC.T
        diff = muc - mu
        SB += Dc.shape[1] * (diff @ diff.T)
    SW /= D.shape[1]
    SB /= D.shape[1]
    s, U = scipy.linalg.eigh(SB, SW)
    return U[:, ::-1][:, :m]

def apply_lda(W, D):
    return W.T @ D


**NOTE:**

- You must calculate the **PCA** or **LDA** strictly on the **Training Data**, and then **apply that exact same calculation** to the **Validation Data**.

## Load and split the project data

The same split must be used for all models, so the dataset is split once and then reused for every experiment.

In [ ]:
D, L = load_project_data('../../../Project/trainData.txt')
(DTR, LTR), (DVAL, LVAL) = split_db(D, L, seed=0)
print('Whole dataset:', D.shape, L.shape)
print('Training set:', DTR.shape, LTR.shape)
print('Validation set:', DVAL.shape, LVAL.shape)
print('Training class counts:', np.bincount(LTR))
print('Validation class counts:', np.bincount(LVAL))


## Full 6-feature models

We evaluate 

- MVG
- Tied Gaussian
- Naive Bayes Gaussian

on the original 6-dimensional feature space, using LLR threshold `0` under uniform priors.

**Note:**

- Having uniform priors just means you are assuming every class has the exact same chance of naturally occurring in the wild

### What Naive Bayes is actually doing — feature by feature

The plots of a Gaussian fitted to each feature of each class independently are exactly what **Naive Bayes** does internally. Naive Bayes factorizes the joint density of all features as a product of 1D densities, one per feature:

$$p(x \mid c) = p(x_1 \mid c) \cdot p(x_2 \mid c) \cdots p(x_6 \mid c)$$

Because of this factorization, the **LLR for Naive Bayes** is simply the **sum of the individual LLRs** for each feature:

$$\text{LLR}(x) = \sum_{i=1}^{6} \log \frac{p(x_i \mid c=1)}{p(x_i \mid c=0)}$$

This means: to understand what Naive Bayes will do, you can look at each feature histogram individually. Naive Bayes just adds those individual scores together. That is why the feature-by-feature analysis directly predicts the classifier's behaviour.

In [ ]:
params_mvg, llr_mvg, pred_mvg, err_mvg = evaluate_model(DTR, LTR, DVAL, LVAL, 'MVG')
params_tied, llr_tied, pred_tied, err_tied = evaluate_model(DTR, LTR, DVAL, LVAL, 'Tied')
params_naive, llr_naive, pred_naive, err_naive = evaluate_model(DTR, LTR, DVAL, LVAL, 'Naive')

results_full = pd.DataFrame([
    ['MVG', err_mvg * 100],
    ['Tied Gaussian', err_tied * 100],
    ['Naive Bayes Gaussian', err_naive * 100],
], columns=['Model', 'Error rate %'])

results_full


### Answer: full-model comparison

On the full 6-feature dataset, MVG obtains **7.00%** error, tied Gaussian obtains **9.30%** error, and Naive Bayes Gaussian obtains **7.20%** error on the validation split.

Among these three Gaussian generative models, **MVG** performs best on this split.

### Why Naive Bayes ≈ MVG on this dataset

The correlation matrices (see below) show that the off-diagonal entries are very small compared to the diagonal. The features are already approximately **independent inside each class** — the within-class covariance is nearly diagonal.

Since MVG estimates a nearly-diagonal covariance anyway, **forcing it to be exactly diagonal** (which is what Naive Bayes does) changes almost nothing. The two models find almost the same decision boundary.

In some cases Naive Bayes does slightly better than MVG because the true correlations should be exactly zero (the features were generated independently). The small non-zero off-diagonal values that MVG estimates are just **finite-sample noise**. Naive Bayes avoids estimating that noise and can be slightly more accurate on small datasets.

## LDA comparison

Comparison of **Tied Gaussian** with **MVG** and **LDA**.

In [ ]:
# m=1 cause we are doing binary
W = compute_lda(DTR, LTR, m=1) 

DTR_LDA = apply_lda(W, DTR)

if DTR_LDA[0, LTR == 1].mean() < DTR_LDA[0, LTR == 0].mean():
    W = -W
    DTR_LDA = apply_lda(W, DTR)

DVAL_LDA = apply_lda(W, DVAL)

# threshold = (DTR_LDA[0, LTR == 0].mean() + DTR_LDA[0, LTR == 1].mean()) / 2.0
# threshold = (DTR[0, LTR == 0].mean() + DTR[0, LTR == 1].mean()) / 2.0
threshold = 0   

pred_lda = np.int32(DVAL_LDA[0] >= threshold)

err_lda = compute_error_rate(pred_lda, LVAL)

print('LDA error rate: %.2f%%' % (err_lda * 100))



### Answer: MVG, tied Gaussian, and LDA

LDA gives **9.20%** error on the same validation split.
Comparing the four methods:
-  **MVG** gives **7.00%**
-  **Tied Gaussian** gives **9.30%**
- **Naive Bayes Gaussian** gives **7.20%**
- **LDA** gives **9.20%** error.


### Why Tied Gaussian and LDA are theoretically the same classifier

Both Tied Gaussian and LDA use **one mean per class** and a **single shared covariance matrix** across all classes. When the LLR threshold is 0 (uniform priors), both produce the same linear decision boundary in feature space.

Any small numerical difference in the error rates (e.g. 9.25% vs 9.30%) comes from differences in the threshold: LDA uses the midpoint of the projected class means (a heuristic), while Tied Gaussian uses threshold = 0 (the theoretically correct threshold under equal priors). These are not exactly the same when the class sizes are slightly unequal in the training split.

Setting `threshold = 0` in the LDA classification step will make both results identical.

### Why Tied Gaussian & LDA have different error rate ?

In theory, a Tied Gaussian generative model and Fisher's Linear Discriminant Analysis (LDA) will find the exact same linear decision boundary. The vector (direction) they use to separate the classes is mathematically identical.

So why the `0.10%` difference in error rate (`9.20%` for LDA vs. `9.30%` for Tied Gaussian)?

The difference lies entirely in how the code chooses the exact threshold (intercept) along that boundary.

1. **Tied Gaussian Threshold**

    For the Tied Gaussian model, you calculate the full multivariate Log-Likelihood Ratio (LLR) and compare it against a strict theoretical threshold of 0 (because of uniform priors).

2. **LDA**

    In the LDA code, the data is first projected down from 6D to a 1-dimensional line. To decide where to split the classes on this 1D line, the code uses a heuristic midpoint threshold between the projected means of the two classes

## Covariance matrices of the two classes

Inspect the class covariance matrices extracted from the:

-  **MVG model** and compare **off-diagonal co-variances** to **diagonal variances**.

In [ ]:
cov0 = abs(np.int16(params_mvg[0][1] * 100))
cov1 = abs(np.int16(params_mvg[1][1] * 100))

# cov0 = params_mvg[0][1] 
# cov1 = params_mvg[1][1]

print('Covariance matrix - class 0')
print("", "__" * 30)

print(cov0)



print('\nCovariance matrix - class 1')
print( "__" * 30)
print(cov1)

### Answer: covariance magnitudes

- In both covariance matrices, the diagonal terms are the feature variances and they are generally larger in scale than many off-diagonal terms.

- The off-diagonal co-variances are not all negligible, so some feature dependence exists, but they are often smaller than the corresponding variances.

### What the covariance matrices tell us

The covariance matrix has:
- **Diagonal entries** = variance of each individual feature (how spread out each feature is on its own)
- **Off-diagonal entries** = covariance between pairs of features (how much two features vary together)

When the off-diagonal entries are close to zero, the features are approximately **uncorrelated inside the class**. This is the condition that makes the **Naive Bayes assumption** approximately valid: Naive Bayes assumes all features are independent inside each class, which is equivalent to assuming a diagonal covariance matrix.

On this dataset both class covariance matrices are approximately diagonal, which is why Naive Bayes works nearly as well as MVG.

## Correlation matrices

"Normalizing co-variances" just means we do a specific math trick: we divide the messy covariance number by the standard deviations of the features.

Correlation coefficients normalize co-variances by the feature standard deviations and make it easier to judge how strongly the features are related.
Doing this mathematically squishes that chaotic covariance number into a strict, standardized scale between -1 and 1. This new, clean number is your Correlation Coefficient.

- 1 = Perfectly correlated (they move up exactly together).

- 1 = Perfectly inversely correlated (when one goes up, the other goes down).

- 0 = Completely independent (Feature 1 does not care at all what Feature 2 is doing).

In [ ]:
def corr_from_cov(C):
    return C / (vcol(C.diagonal() ** 0.5) * vrow(C.diagonal() ** 0.5))

corr0 = corr_from_cov(cov0)
corr1 = corr_from_cov(cov1)

# corr0 = np.int16(abs(corr_from_cov(cov0) * 100))
# corr1 = np.int16(abs(corr_from_cov(cov1) * 100))

print('Correlation matrix - class 0')
print("", "__" * 30)

print(corr0)

print('\nCorrelation matrix - class 1')
print("", "__" * 30)

print(corr1)

### Answer: are the features strongly or weakly correlated?

The features appear mostly **weakly to moderately correlated**, not extremely correlated, because the off-diagonal correlation coefficients are generally well below ±1.

On this split, the largest absolute off-diagonal correlation is about **0.049**, which indicates that correlations exist but are not dominant across all pairs.

This is consistent with the Naive Bayes results:

- The independence assumption is not exactly true, but it is not violated so strongly that the model completely breaks down.

### Why near-zero correlations validate the Naive Bayes assumption

The correlation matrix normalizes the covariance so all diagonal entries become 1. Off-diagonal entries close to 0 mean the two features are approximately independent.

Naive Bayes assumes the features are **conditionally independent given the class**, which is exactly the assumption that all off-diagonal entries of the within-class correlation matrix are 0. Since the data approximately satisfies this condition, Naive Bayes is approximately correct here — which explains why its error rate is close to MVG.

## Gaussian assumption from Laboratory 5

The project asks to interpret the results in light of the single-feature Gaussian fitting from the previous laboratory, which corresponds to the Naive Bayes view of the data.

1. What was the "single-feature Gaussian fitting" in Lab 5?
Think back to Laboratory 5. You likely took your dataset and looked at just one feature at a time (for example, just Feature 1). You probably plotted a histogram of Feature 1, and then mathematically drew a 1-Dimensional bell curve (a Gaussian distribution) right over the top of it to see if the data fit the shape of the curve. You evaluated them completely independently.

2. Why does that correspond to the "Naive Bayes view"?
As we discussed earlier, the Naive Bayes model stubbornly assumes that every single feature is 100% independent. It completely ignores the off-diagonal covariance numbers (the relationships between features).

Because it ignores relationships, Naive Bayes is mathematically identical to just taking those six separate, 1-Dimensional single-feature bell curves from Lab 5 and multiplying them together. It views your 6-dimensional dataset not as a complex, interconnected web, but simply as six isolated 1D features sitting next to each other.

What the instructor actually wants you to write:
Your instructor is asking you to connect the dots between the visual graphs you made in Lab 5 and the accuracy numbers you are seeing now. They want you to ask yourself:

"When I looked at the single features in Lab 5, did they actually look like perfect bell curves?"

If the answer is Yes: Then the Gaussian assumption is solid, and the models should work beautifully.
If the answer is No: (e.g., the data was skewed to one side, or had two peaks instead of one), then the foundational assumption of your model is flawed.

### Answer: goodness of the Gaussian assumption

The Gaussian assumption is not equally accurate for all six features.

- Some features are reasonably well approximated by a single Gaussian, while others show shapes that are less symmetric or less bell-shaped so the Gaussian fit is only approximate there.

- This helps explain why Gaussian generative classifiers work reasonably well overall but are not perfect models of the underlying feature distributions.

### Features 5 & 6 — bad fit but still useful

For features 5 and 6, the Gaussian fit is clearly **bad**: each class has multiple clusters (you saw 4 clusters total in the scatter plot), so the real distribution is multi-modal. A single Gaussian cannot capture this shape — the red curve sits somewhere in the middle and does not match the histogram peaks.

Despite this, the classifier **still extracts information** from these features. Here is why:

When you plot both class Gaussian curves on the same axis, they cross at **two points**. Between those crossing points, one class density is higher → LLR > 0 → predict class 1. Outside the crossing points, the other class density is higher → LLR < 0 → predict class 0. Even with an inaccurate Gaussian shape, the model captures enough structure to produce a useful LLR signal.

> **Key insight from the professor:** *'A model that has a bad fit may still be useful when it is easy to build and use to extract information. If you remove features 5 and 6 from the classifier, you will actually get WORSE results.'*

Good estimation (good fit) does **not** guarantee good classification. And a bad fit does **not** mean a useless model. The fit tells you how accurately the model describes the data distribution. The classification performance tells you how well the model places the decision boundary. These are independent.

## Using only features 1 to 4

The project asks to discard the last two features and repeat the three Gaussian classifiers.

In [ ]:
idx_1_4 = np.array([0, 1, 2, 3])
DTR_1_4 = DTR[idx_1_4, :]
DVAL_1_4 = DVAL[idx_1_4, :]
_, _, _, err_mvg_1_4 = evaluate_model(DTR_1_4, LTR, DVAL_1_4, LVAL, 'MVG')
_, _, _, err_tied_1_4 = evaluate_model(DTR_1_4, LTR, DVAL_1_4, LVAL, 'Tied')
_, _, _, err_naive_1_4 = evaluate_model(DTR_1_4, LTR, DVAL_1_4, LVAL, 'Naive')

pd.DataFrame([
    ['MVG', err_mvg_1_4 * 100],
    ['Tied Gaussian', err_tied_1_4 * 100],
    ['Naive Bayes Gaussian', err_naive_1_4 * 100],
], columns=['Model', 'Error rate %'])


### Answer: discarding the last two features

Using only features 1 to 4, MVG gives **7.95%** error, tied Gaussian gives **9.50%** error, and Naive Bayes Gaussian gives **7.65%** error.

This shows that discarding the last two features does **not improve** the best validation result on this split. 

Therefore, even if the Gaussian modeling assumption is less accurate for the last two features, those features can still provide useful discriminative information.

### Why removing features 5 & 6 increases the error rate

Even though the Gaussian model fits features 5 and 6 poorly, those features still carry **discriminative information**. The LLR scores for features 5 and 6 — even from a bad model — contribute something useful when summed with the scores from the other features in the Naive Bayes combination.

Removing them throws away that information entirely. The result is a higher error rate (7.95% vs 7.00%), proving that the classifier was using those features productively despite the model mismatch.

## Features 1-2 versus features 3-4

The project asks to classify using only features 1-2 jointly and only features 3-4 jointly, and to compare MVG with tied Gaussian.

In [ ]:
idx_1_2 = np.array([0, 1])
idx_3_4 = np.array([2, 3])

_, _, _, err_mvg_1_2 = evaluate_model(DTR[idx_1_2, :], LTR, DVAL[idx_1_2, :], LVAL, 'MVG')
_, _, _, err_tied_1_2 = evaluate_model(DTR[idx_1_2, :], LTR, DVAL[idx_1_2, :], LVAL, 'Tied')
_, _, _, err_mvg_3_4 = evaluate_model(DTR[idx_3_4, :], LTR, DVAL[idx_3_4, :], LVAL, 'MVG')
_, _, _, err_tied_3_4 = evaluate_model(DTR[idx_3_4, :], LTR, DVAL[idx_3_4, :], LVAL, 'Tied')

pd.DataFrame([
    ['Features 1-2', err_mvg_1_2 * 100, err_tied_1_2 * 100],
    ['Features 3-4', err_mvg_3_4 * 100, err_tied_3_4 * 100],
], columns=['Subset', 'MVG error %', 'Tied error %'])


### Answer: features 1-2 and features 3-4

For features 1-2, MVG gives **36.50%** error and tied Gaussian gives **49.45%** error, so **MVG** performs better on this subset.

For features 3-4, MVG gives **9.45%** error and tied Gaussian gives **9.40%** error, so **Tied Gaussian** performs better there.

This is coherent with the project discussion: MVG is more flexible when covariance structure differs by class, whereas the tied model is more appropriate when the classes mainly differ in mean and have more similar covariance structure.

### Why features 1-2 are nearly useless and features 3-4 work well

**Features 1 & 2:** The two classes have almost the **same mean**. The between-class variance $S_B$ is near zero for these features. Even with a perfect model, the LLR is close to zero everywhere — the classifier cannot distinguish the classes and produces ~50% error (36.50% here because MVG also uses the different variances).

**Features 3 & 4:** The class means are **well separated**, and the within-class variance is approximately the same for both classes. The between-class variance $S_B$ is large relative to the within-class variance $S_W$, giving a high LDA ratio. The classifier works well (9.45% error).

This also explains why LDA places almost no weight on features 1 and 2: when the within-class variance is the same in every direction, LDA selects the direction that connects the class means — which is dominated by features 3 and 4.

## PCA as pre-processing

Finally, the project asks to apply PCA before the three Gaussian classifiers and evaluate whether dimensionality reduction helps.

In [ ]:
pca_results = []
for m in range(1, DTR.shape[0] + 1):
    P = compute_pca(DTR, m)
    DTR_pca = apply_pca(P, DTR)
    DVAL_pca = apply_pca(P, DVAL)
    _, _, _, err_mvg_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'MVG')
    _, _, _, err_tied_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'Tied')
    _, _, _, err_naive_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'Naive')
    pca_results.append([m, 'MVG', err_mvg_pca * 100])
    pca_results.append([m, 'Tied Gaussian', err_tied_pca * 100])
    pca_results.append([m, 'Naive Bayes Gaussian', err_naive_pca * 100])

pca_df = pd.DataFrame(pca_results, columns=['m', 'Model', 'Error rate %'])
pca_df.pivot(index='m', columns='Model', values='Error rate %')


In [ ]:
pca_results = []

# 1. Evaluate the 3 models WITHOUT PCA (Raw Data)
_, _, _, err_mvg_raw = evaluate_model(DTR, LTR, DVAL, LVAL, 'MVG')
_, _, _, err_tied_raw = evaluate_model(DTR, LTR, DVAL, LVAL, 'Tied')
_, _, _, err_naive_raw = evaluate_model(DTR, LTR, DVAL, LVAL, 'Naive')

# Append them to our list and explicitly mark them as 'No' for PCA
pca_results.append(['All 6 Features', 'No', 'MVG', err_mvg_raw * 100])
pca_results.append(['All 6 Features', 'No', 'Tied Gaussian', err_tied_raw * 100])
pca_results.append(['All 6 Features', 'No', 'Naive Bayes Gaussian', err_naive_raw * 100])

# 2. Evaluate the models WITH PCA (Inside your loop)
for m in range(1, DTR.shape[0] + 1):
    P = compute_pca(DTR, m)
    DTR_pca = apply_pca(P, DTR)
    DVAL_pca = apply_pca(P, DVAL)
    
    _, _, _, err_mvg_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'MVG')
    _, _, _, err_tied_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'Tied')
    _, _, _, err_naive_pca = evaluate_model(DTR_pca, LTR, DVAL_pca, LVAL, 'Naive')
    
    # Append them to our list and explicitly mark them as 'Yes' for PCA
    pca_results.append([m, 'Yes', 'MVG', err_mvg_pca * 100])
    pca_results.append([m, 'Yes', 'Tied Gaussian', err_tied_pca * 100])
    pca_results.append([m, 'Yes', 'Naive Bayes Gaussian', err_naive_pca * 100])

# 3. Create the DataFrame with the new column
pca_df = pd.DataFrame(pca_results, columns=['m', 'Has PCA?', 'Model', 'Error rate %'])

# 4. Display the clean pivot table
pca_df.pivot(index=['Has PCA?', 'm'], columns='Model', values='Error rate %')

### Answer: Is PCA effective, and what is the best overall model?

In these experiments, PCA is **not especially effective overall** for the Gaussian models on this validation split.

The best PCA-based configuration is **MVG** with **m = 6**, giving **7.00%** error.

Overall, the best validation result across the experiments is **MVG** with **7.00%** error.

### Why PCA before Naive Bayes INCREASES the error rate

This is the most important warning from the professor about PCA and generative models.

You observe: error goes from **7.2% → 8.9%** when you apply PCA (6 dimensions) before Naive Bayes. Intuitively this seems wrong — PCA uncorrelates features, and Naive Bayes assumes uncorrelated features. Should they not work well together?

The mistake is in that reasoning. Here is the correct explanation:

- **PCA** diagonalizes the **total covariance matrix** $S_T$ — computed on ALL samples together, ignoring class labels.
- **Naive Bayes** needs the **within-class covariance** $S_{W,c}$ of each individual class to be diagonal.

These are two completely different matrices. Making $S_T$ diagonal does NOT make $S_{W,c}$ diagonal.

**What actually happens:** Before PCA, the within-class ovals are approximately axis-aligned (features independent inside each class → Naive Bayes assumption holds). PCA rotates everything to align with the combined diagonal direction of the total dataset. After the rotation, each class oval is now **tilted** — within-class features are now correlated. Naive Bayes still blindly assumes independence, so it is now modeling the data incorrectly. The error rate increases.

> *'PCA uncorrelates the dataset features seen as a whole, but Naive Bayes assumes the features of each class are uncorrelated. These two are not necessarily related. I could even have the effect that PCA uncorrelates the features as a whole, but the rotation introduces correlations inside each class.'* — Professor

**For MVG and Tied:** PCA with all 6 dimensions is just a rotation. MVG adapts its full covariance matrix to the new orientation, so the result is exactly the same. No change in error rate.

## Final discussion

On the full dataset, the generative Gaussian models give different results because they impose different covariance assumptions: MVG uses class-specific full covariances, tied Gaussian shares one covariance across classes, and Naive Bayes keeps only diagonal variances. On this split, the corresponding validation errors are MVG **7.00%**, tied Gaussian **9.30%**, Naive Bayes Gaussian **7.20%**, and LDA **9.20%**.

The covariance and correlation analysis indicates that the features are not completely independent, but many dependencies are only weak to moderate.This explains why Naive Bayes can still be competitive even though its independence assumption is not fully correct.

The single-feature Gaussian analysis from Laboratory 5 suggests that the Gaussian assumption is not equally accurate for all six features.Nevertheless, even features that are not perfectly Gaussian can still carry useful discriminative information, which is why removing them does not necessarily improve classification.

Overall, the best result in this notebook is **MVG** with **7.00%** validation error.